# AMPS II · Week 12 — Lab
## Reactive flow: a tubular reactor — axial dispersion, laminar flow, SUPG, the Graetz problem

| | |
|---|---|
| **Lecture sections** | Part A (the model, its limits, the mixing cup) · Part B (axisymmetric weak form, Danckwerts, SUPG) · Part C (verification against exact limits) · Part D (Navier–Stokes, LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | compute the exact limits in NumPy · reproduce the dispersion model to 8 digits and compare two inlet conditions · check a laminar-flow reactor against the segregated-flow integral · see SUPG fail when its residual is incomplete · reproduce the Graetz series and $Sh_\infty=3.657$ · judge an LLM's reactor script |

**Reminder.** `%%fenicsx` cells are separate MPI programs; ordinary cells have NumPy and matplotlib only. The model module `reactor.py` is written once with `%%writefile` (§1). All cells run with `-np 2`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The model module  *(lecture §A–D)*
`reactor.py`: `tube_mesh(L, nz, nr, grade)` in $(z,r)$ (lengths in units of the radius $R$), `velocity(msh, "plug" | "poiseuille")` (mean 1), `Flow` (steady axisymmetric Navier–Stokes, uniform inlet),
`Species(msh, u, Pe, Da, order, inlet="danckwerts" | "dirichlet", wall="inert" | "reactive", supg, supg_reaction, hkind)` with `.solve()`, `.mixing_cup()`, `.area_mean()`, `.profile(z)`,
`.mixing_cup_at(zs)`, `.sherwood(zs)`; `probe`; exact references `danckwerts`, `dirichlet_dispersion`, `segregated`, `taylor_aris`, `graetz`. Every integral carries the weight $r$. Run the cell.

In [ ]:
%%writefile /tmp/fnx/reactor.py
"""reactor.py — a tubular reactor, axisymmetric (z, r), DOLFINx 0.11 (AMPS II, Week 12).

Coordinates x[0] = z (axial, 0..L), x[1] = r (radial, 0..1, axis at r = 0); lengths in units of the tube radius R,
velocities in units of the mean velocity U. Every integral carries the weight r (axisymmetry; the 2*pi cancels).
    species:   u . grad c = (1/Pe) lap_axi c - Da c^n          Pe = U R / D, Da = k R c0^(n-1) / U
    flow:      (u . grad) u = -grad p + (1/Re) (lap_axi u - u_r/r^2 e_r),  div_axi u = 0,   Re = U R / nu
Inlet z = 0 (Danckwerts or Dirichlet), outlet z = L (no diffusive flux / do-nothing), wall r = 1 (inert or c = 0), axis r = 0."""
import gc
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem

comm = MPI.COMM_WORLD
gc.disable()                                   # PETSc destructors are collective: collect at the same point on all ranks
def collect():
    gc.collect()
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)
LU = {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}
NEWTON = dict(LU, snes_type="newtonls", snes_linesearch_type="none", snes_rtol=1e-10, snes_atol=1e-11, snes_stol=0.0, snes_max_it=30)

def tube_mesh(L=10.0, nz=100, nr=10, grade=0.0):
    """[0, L] x [0, 1] in (z, r), quadrilaterals; grade > 0 clusters cells towards the wall r = 1 (r -> 1 - (1 - s)^(1 + grade))"""
    msh = mesh.create_rectangle(comm, [[0, 0], [L, 1]], [nz, nr], mesh.CellType.quadrilateral)
    if grade > 0:
        x = msh.geometry.x; x[:, 1] = 1 - (1 - x[:, 1]) ** (1 + grade)
    return msh

def _L(msh):
    return comm.allreduce(msh.geometry.x[:, 0].max(), op=MPI.MAX)

def velocity(msh, kind="poiseuille", deg=2):
    """prescribed velocity (u_z, u_r) with mean 1: "plug" (u_z = 1) or "poiseuille" (u_z = 2 (1 - r^2)); a P2 vector Function"""
    V = fem.functionspace(msh, ("Lagrange", deg, (2,))); u = fem.Function(V)
    if kind == "plug":
        u.interpolate(lambda x: np.vstack([np.ones(x.shape[1]), np.zeros(x.shape[1])]))
    else:
        u.interpolate(lambda x: np.vstack([2 * (1 - x[1] ** 2), np.zeros(x.shape[1])]))
    return u

class Flow:
    """steady axisymmetric Navier-Stokes in the tube: uniform inlet u = (1, 0), no slip at r = 1, u_r = 0 on the axis, do-nothing outlet"""
    count = 0
    def __init__(self, msh, Re=10.0, inlet="uniform"):
        collect(); Flow.count += 1
        self.msh = msh; c = msh.basix_cell(); self.L = _L(msh)
        W = self.W = fem.functionspace(msh, basix.ufl.mixed_element([basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)]))
        self.w = fem.Function(W); u, p = ufl.split(self.w); v, q = ufl.TestFunctions(W)
        self.Re = fem.Constant(msh, float(Re)); r = ufl.SpatialCoordinate(msh)[1]
        div = lambda a: a[0].dx(0) + a[1].dx(1) + a[1] / r
        F = (ufl.inner(ufl.grad(u) * u, v) + 1 / self.Re * (ufl.inner(ufl.grad(u), ufl.grad(v)) + u[1] * v[1] / r ** 2)
             - p * div(v) - q * div(u)) * r * ufl.dx
        V, _ = W.sub(0).collapse(); Vr, _ = W.sub(0).sub(1).collapse()
        fdim = 1
        inl = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], 0))
        wall = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[1], 1))
        axis = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[1], 0))
        uin = fem.Function(V)
        if inlet == "uniform":
            uin.interpolate(lambda x: np.vstack([np.ones(x.shape[1]), np.zeros(x.shape[1])]))
        else:
            uin.interpolate(lambda x: np.vstack([2 * (1 - x[1] ** 2), np.zeros(x.shape[1])]))
        self.bcs = [fem.dirichletbc(fem.Function(V), fem.locate_dofs_topological((W.sub(0), V), fdim, wall), W.sub(0)),
                    fem.dirichletbc(uin, fem.locate_dofs_topological((W.sub(0), V), fdim, inl), W.sub(0)),
                    fem.dirichletbc(fem.Function(Vr), fem.locate_dofs_topological((W.sub(0).sub(1), Vr), fdim, axis), W.sub(0).sub(1))]
        self.pb = NonlinearProblem(F, self.w, bcs=self.bcs, petsc_options_prefix=f"fl{Flow.count}_", petsc_options=NEWTON)

    def solve(self, Re=None):
        if Re is not None:
            self.Re.value = Re
        self.pb.solve(); s = self.pb.solver
        return int(s.getConvergedReason()), int(s.getIterationNumber())

    def u(self):
        return self.w.sub(0).collapse()

    def p(self):
        return self.w.sub(1).collapse()

class Species:
    """u . grad c = (1/Pe) lap_axi c - Da c^n, axisymmetric; inlet "danckwerts" or "dirichlet" (c = 1), wall "inert" or "reactive" (c = 0)"""
    count = 0
    def __init__(self, msh, u, Pe=100.0, Da=1.0, order=1, inlet="danckwerts", wall="inert", supg=True, supg_reaction=True, hkind="streamline", deg=2):
        collect(); Species.count += 1
        self.msh = msh; self.u = u; self.L = _L(msh); self.order = order
        S = self.S = fem.functionspace(msh, ("Lagrange", deg)); self.c = fem.Function(S); w = ufl.TestFunction(S)
        self.Pe, self.Da = fem.Constant(msh, float(Pe)), fem.Constant(msh, float(Da))
        X = ufl.SpatialCoordinate(msh); r = X[1]; c = self.c; n = ufl.FacetNormal(msh); self.r = r
        fdim = 1
        inl = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], 0))
        out = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], self.L))
        wl = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[1], 1))
        tags = np.concatenate([inl, out, wl]); vals = np.r_[np.full(len(inl), 1), np.full(len(out), 2), np.full(len(wl), 3)].astype(np.int32)
        o = np.argsort(tags); tags, vals = tags[o], vals[o]
        self.ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, fdim, tags, vals))
        rate = c if order == 1 else c * c if order == 2 else c ** order
        F = (ufl.dot(u, ufl.grad(c)) * w + 1 / self.Pe * ufl.dot(ufl.grad(c), ufl.grad(w)) + self.Da * rate * w) * r * ufl.dx
        self.bcs = []
        if inlet == "danckwerts":                                                     # u c - (1/Pe) dc/dz = u * 1 at z = 0
            F += u[0] * (c - 1.0) * w * r * self.ds(1)
        else:
            one = fem.Function(S); one.x.array[:] = 1.0
            self.bcs.append(fem.dirichletbc(one, fem.locate_dofs_topological(S, fdim, inl)))
        if wall == "reactive":
            self.bcs.append(fem.dirichletbc(fem.Function(S), fem.locate_dofs_topological(S, fdim, wl)))
        if supg:
            un = ufl.sqrt(ufl.dot(u, u) + 1e-30)
            if hkind == "streamline":                                                   # cell length along u: |u| / |J^-1 u|
                Ku = ufl.JacobianInverse(msh) * u; h = un / ufl.sqrt(ufl.dot(Ku, Ku) + 1e-30)
            else:                                                                       # the cell's diameter (diagonal)
                h = ufl.CellDiameter(msh)
            Peh = un * h * self.Pe / (2 * deg)
            tau = h / (2 * deg * un) * (1 / ufl.tanh(Peh) - 1 / Peh)                     # SUPG parameter (cell Peclet with h/deg)
            lap = ufl.div(ufl.grad(c)) + c.dx(1) / r                                    # axisymmetric Laplacian
            res = ufl.dot(u, ufl.grad(c)) - 1 / self.Pe * lap + (self.Da * rate if supg_reaction else 0)
            F += tau * res * ufl.dot(u, ufl.grad(w)) * r * ufl.dx
        self.F = F
        self.pb = NonlinearProblem(F, c, bcs=self.bcs, petsc_options_prefix=f"sp{Species.count}_", petsc_options=NEWTON)
        self.c.x.array[:] = 1.0 if wall == "inert" else 0.5

    def solve(self, Pe=None, Da=None):
        if Pe is not None:
            self.Pe.value = Pe
        if Da is not None:
            self.Da.value = Da
        self.pb.solve(); s = self.pb.solver
        return int(s.getConvergedReason()), int(s.getIterationNumber())

    # ── quantities ─────────────────────────────────────────────────
    def _int(self, e, m):
        return comm.allreduce(fem.assemble_scalar(fem.form(e * m)), op=MPI.SUM)

    def mixing_cup(self):
        """outlet concentration weighted by the flow: int u_z c r dr / int u_z r dr at z = L"""
        return self._int(self.u[0] * self.c * self.r, self.ds(2)) / self._int(self.u[0] * self.r, self.ds(2))

    def area_mean(self):
        """outlet concentration averaged over the cross-section: int c r dr / int r dr"""
        return self._int(self.c * self.r, self.ds(2)) / self._int(self.r, self.ds(2))

    def profile(self, z, n=201):
        """r and c(z, r) on a radial line (every rank gets all values)"""
        P = np.zeros((n, 3)); P[:, 0] = z; P[:, 1] = np.linspace(0, 1, n)
        return P[:, 1], probe(self.msh, self.c, P)

    def mixing_cup_at(self, zs, n=401):
        """mixing-cup concentration at axial positions zs: int u_z c r dr / int u_z r dr on radial lines (Simpson)"""
        Uz = self.u.sub(0).collapse() if hasattr(self.u, "sub") else None
        out = []
        for z in np.atleast_1d(zs):
            P = np.zeros((n, 3)); P[:, 0] = z; P[:, 1] = np.linspace(0, 1, n)
            cr = probe(self.msh, self.c, P); ur = probe(self.msh, Uz, P); r = P[:, 1]
            wts = np.ones(n); wts[1:-1:2] = 4; wts[2:-1:2] = 2
            out.append(np.sum(wts * ur * cr * r) / np.sum(wts * ur * r))
        return np.array(out)

    def sherwood(self, zs, dz=0.05):
        """local Sherwood number (diameter-based) from the decay of the mixing-cup concentration, reactive wall:
        Sh = -Pe d ln(c_mix)/dz (axial diffusion neglected in the balance)"""
        zs = np.atleast_1d(zs); a = self.mixing_cup_at(zs - dz); b = self.mixing_cup_at(zs + dz)
        return -float(self.Pe.value) * (np.log(b) - np.log(a)) / (2 * dz)

def probe(msh, f, P):
    """values of a scalar Function at points P (n x 3) — every rank gets all values"""
    from dolfinx import geometry
    nloc = msh.topology.index_map(2).size_local
    cand = geometry.compute_colliding_cells(msh, geometry.compute_collisions_points(geometry.bb_tree(msh, 2), P), P)
    out, cnt = np.zeros(len(P)), np.zeros(len(P)); idx, cells = [], []
    for j in range(len(P)):
        own = [c for c in cand.links(j) if c < nloc]
        if own:
            idx.append(j); cells.append(own[0])
    if idx:
        val = np.asarray(f.eval(P[idx], np.array(cells, dtype=np.int32))).reshape(len(idx), -1)
        out[idx] = val[:, 0]; cnt[idx] = 1
    return comm.allreduce(out, op=MPI.SUM) / np.maximum(comm.allreduce(cnt, op=MPI.SUM), 1)

# exact references ─────────────────────────────────────────────────────────────
def danckwerts(PeL, DaL):
    """plug flow with axial dispersion, 1st order, Danckwerts inlet: outlet c (Pe_L = U L / D, Da_L = k L / U)"""
    a = np.sqrt(1 + 4 * DaL / PeL)
    return 4 * a * np.exp(PeL / 2 * (1 - a)) / ((1 + a) ** 2 - (1 - a) ** 2 * np.exp(-a * PeL))

def dirichlet_dispersion(PeL, DaL):
    """plug flow with axial dispersion, 1st order, c = 1 at the inlet, dc/dz = 0 at the outlet: outlet c"""
    m1 = PeL / 2 * (1 + np.sqrt(1 + 4 * DaL / PeL)); m2 = PeL / 2 * (1 - np.sqrt(1 + 4 * DaL / PeL))
    # c = A e^{m1 s} + B e^{m2 s}, s in [0, 1]: A + B = 1, A m1 e^{m1} + B m2 e^{m2} = 0
    B = 1 / (1 - m2 * np.exp(m2 - m1) / m1); A = 1 - B
    return A * np.exp(m1) + B * np.exp(m2)

def taylor_aris(Pe, L, Da):
    """laminar flow with radial diffusion, long tube: plug flow with the Taylor-Aris dispersion D_eff = D (1 + Pe^2 / 48),
    Pe = U R / D; returns the outlet c of the Danckwerts dispersion model"""
    Pe_eff = Pe / (1 + Pe ** 2 / 48)
    return danckwerts(Pe_eff * L, Da * L)

def segregated(DaL, order=1, n=200001):
    """laminar (Poiseuille) flow without diffusion: every streamline a plug-flow reactor with residence time L / u(r); mixing cup"""
    r = np.linspace(0, 1, n)[:-1] + 0.5 / n; u = 2 * (1 - r ** 2); t = DaL / u
    c = np.exp(-t) if order == 1 else 1 / (1 + t)
    return np.sum(u * c * 2 * r) / n

def graetz(nev=10, N=800):
    """Graetz problem: 2 (1 - r^2) dc/dzeta = (1/r)(r c_r)_r, c(1) = 0, c(r, 0) = 1, zeta = z / Pe (axial diffusion neglected).
    Finite volumes in r -> eigenvalues w_k and mixing-cup weights m_k: c_mix(zeta) = sum_k m_k exp(-w_k zeta); Sh_inf = w_1 = 3.657"""
    h = 1.0 / N; r = (np.arange(N) + 0.5) * h; rp = r + h / 2; rm = r - h / 2
    A = np.zeros((N, N))
    for i in range(N):
        A[i, i] = -(rp[i] + rm[i]) / h ** 2
        if i > 0:
            A[i, i - 1] = rm[i] / h ** 2
        if i < N - 1:
            A[i, i + 1] = rp[i] / h ** 2
    A[N - 1, N - 1] = -(rm[N - 1] / h ** 2 + rp[N - 1] / (h / 2) / h)      # c = 0 at r = 1 (half cell)
    Bm = np.diag(2 * (1 - r ** 2) * r)
    from numpy.linalg import eig
    w, V = eig(np.linalg.solve(Bm, -A))
    o = np.argsort(w.real); w = w.real[o][:nev]; V = V.real[:, o][:, :nev]
    # coefficients for c(r, 0) = 1 (weight 2(1 - r^2) r)
    wt = 2 * (1 - r ** 2) * r * h
    coef = np.array([np.sum(wt * V[:, k]) / np.sum(wt * V[:, k] ** 2) for k in range(nev)])
    # mixing cup c(zeta) = sum coef_k <u phi_k> exp(-w_k zeta) / <u>, <u> = int 2(1-r^2) r dr = 1/2
    mc = np.array([np.sum(wt * V[:, k]) for k in range(nev)]) * coef / 0.5
    return w, mc

## 2 · The exact limits in NumPy  *(lecture §A2)*
(i) Plug flow with axial dispersion, first order, Danckwerts inlet: $c_\text{out}=\dfrac{4a\,e^{Pe_L/2}}{(1+a)^2e^{aPe_L/2}-(1-a)^2e^{-aPe_L/2}}$, $a=\sqrt{1+4Da_L/Pe_L}$.
(ii) Laminar flow without diffusion (segregated): every streamline is a plug-flow reactor with residence time $L/u(r)$, $u=2(1-r^2)$: $\bar c=\int_0^1 u\,e^{-Da_L/u}\,2r\,dr$.
(iii) The Graetz problem (reactive wall, no axial diffusion): $c_\text{mix}(\zeta)=\sum_k m_ke^{-w_k\zeta}$, $\zeta=z/(R\,Pe)$, $w_1=Sh_\infty$.

In [ ]:
import numpy as np
def danckwerts(PeL, DaL):
    a = np.sqrt(1 + 4 * DaL / PeL)
    return 4 * a * np.exp(PeL / 2 * (1 - a)) / ((1 + a) ** 2 - (1 - a) ** 2 * np.exp(-a * PeL))
r = (np.arange(200000) + 0.5) / 200000; u = 2 * (1 - r ** 2)
seg = lambda DaL: np.sum(u * np.exp(-DaL / u) * 2 * r) / 200000
for PeL in (2, 20, 200, 2000):
    print(f"Pe_L {PeL:5d}, Da_L = 2: dispersion model {danckwerts(PeL, 2):.6f}  (plug flow {np.exp(-2):.6f})")
print(f"Da_L = 1: segregated laminar {seg(1):.6f}, plug flow {np.exp(-1):.6f}")
# Graetz eigenproblem by finite volumes in r: (r c')' = -w 2 (1 - r^2) r c, c(1) = 0
N = 800; h = 1 / N; rc = (np.arange(N) + 0.5) * h; rp, rm = rc + h / 2, rc - h / 2
A = np.diag(-(rp + rm) / h ** 2) + np.diag(rm[1:] / h ** 2, -1) + np.diag(rp[:-1] / h ** 2, 1); A[-1, -1] = -(rm[-1] + 2 * rp[-1]) / h ** 2
B = 2 * (1 - rc ** 2) * rc
w, V = np.linalg.eig(-A / B[:, None]); o = np.argsort(w.real); w, V = w.real[o][:5], V.real[:, o][:, :5]
wt = B * h; m = np.array([np.sum(wt * V[:, k]) ** 2 / np.sum(wt * V[:, k] ** 2) for k in range(5)]) / 0.5
print("Graetz eigenvalues w_k:", np.round(w, 4), " mixing-cup weights m_k:", np.round(m, 4), " Sh_inf = w_1 =", round(w[0], 4))

> **✏️ Your turn (3 min).** (i) Why does the dispersion model approach plug flow as $Pe_L\to\infty$, and the segregated integral lie **above** plug flow? (ii) Which of the three limits would you use to check a code for a
> 1 m tube of radius 5 mm with water ($D=10^{-9}$ m²/s) at $U=1$ cm/s?

> *your answer*

## 3 · Plug flow with axial dispersion: Danckwerts or Dirichlet inlet  *(lecture §B2, §C1)*
$u=1$, $L=10$, $Da_L=2$, three Péclet numbers; the inlet condition either Danckwerts ($u\,c-\frac1{Pe}\partial_zc=u$, a Robin term in the weak form) or $c=1$.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from reactor import *

msh = tube_mesh(10, 100, 4); u = velocity(msh, "plug")
for Pe in (0.2, 2.0, 20.0):
    out = {}
    for inlet in ("danckwerts", "dirichlet"):
        sp = Species(msh, u, Pe=Pe, Da=0.2, inlet=inlet, supg=False); r = sp.solve(); out[inlet] = sp.mixing_cup()
    say(f"Pe_L {Pe * 10:5.0f}: Danckwerts FE {out['danckwerts']:.8f} (exact {danckwerts(Pe * 10, 2):.8f}) | Dirichlet FE {out['dirichlet']:.8f} (exact {dirichlet_dispersion(Pe * 10, 2):.8f}) | plug {np.exp(-2):.6f}")

> **✏️ Your turn (5 min).** (i) Which inlet condition is physical, and how large is the difference at $Pe_L=2$? (ii) At $Pe_L=200$ the Dirichlet FE value is off its own exact solution in the 3rd digit, the Danckwerts one is not. Why?

> *your answer*

## 4 · Laminar flow without diffusion: the mixing cup  *(lecture §A3, §C2)*
Poiseuille flow, $Pe=10^8$ (no diffusion), $Da_L=1$, inert wall. Compare the outlet with the segregated integral, the mixing cup with the area mean, and look at the field.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from reactor import *

msh = tube_mesh(10, 100, 20); u = velocity(msh, "poiseuille")
sp = Species(msh, u, Pe=1e8, Da=0.1); r = sp.solve()
cmin = comm.allreduce(sp.c.x.array.min(), op=MPI.MIN); cmax = comm.allreduce(sp.c.x.array.max(), op=MPI.MAX)
say(f"SNES {r}; mixing cup {sp.mixing_cup():.6f} (segregated {segregated(1.0):.6f}); area mean {sp.area_mean():.6f}; c in [{cmin:.4f}, {cmax:.4f}]")
rr, cr = sp.profile(10.0)
if comm.rank == 0:
    np.savez("/tmp/fnx/lfr.npz", r=rr, c=cr)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
d = np.load("/tmp/fnx/lfr.npz"); r = d["r"]
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(r, d["c"], "o", ms=3, mfc="none", label="FE outlet"); ax.plot(r, np.exp(-1 / np.maximum(2 * (1 - r ** 2), 1e-300)), "k-", lw=1, label="exp(-Da L / u(r))")
ax.set(xlabel="r / R", ylabel="c at z = L"); ax.legend(); ax.grid(alpha=0.3); plt.show()

> **✏️ Your turn (5 min).** (i) Which outlet average does a sample taken from the outflow measure, and why do the two differ by so much here? (ii) The field has $c<0$ somewhere although the mixing cup is right to 6 digits. Where, and why does it not matter for the mixing cup?

> *your answer*

## 5 · SUPG: what the residual must contain  *(lecture §B3, §C3)*
Plug flow, $Pe=10^6$ (so that $e^{-Da\,z}$ is exact to $10^{-4}$), $Da=20$ ($L=4$): a decay length $1/Da=0.05$ against cells of $0.2\ldots0.05$. Galerkin, SUPG with the full residual, and SUPG whose residual omits the reaction term.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from reactor import *

for nz in (20, 40, 80):
    msh = tube_mesh(4.0, nz, 2); u = velocity(msh, "plug"); P = np.zeros((401, 3)); P[:, 0] = np.linspace(0, 4, 401); P[:, 1] = 0.5; out = []
    for name, supg, sr in (("Galerkin", False, True), ("SUPG", True, True), ("SUPG without reaction", True, False)):
        sp = Species(msh, u, Pe=1e6, Da=20.0, supg=supg, supg_reaction=sr); sp.solve()
        out.append(f"{name} {np.abs(probe(msh, sp.c, P) - np.exp(-20 * P[:, 0])).max():.2e}")
    say(f"nz {nz:3d}: max |c - exp(-Da z)|: " + " | ".join(out))

> **✏️ Your turn (5 min).** (i) Read off the order of each column. (ii) What does the incomplete residual add to the equation, and why does it look harmless on a fine mesh?

> *your answer*

## 6 · The Graetz problem: a reactive wall  *(lecture §C4)*
Poiseuille flow, wall $c=0$ (instantaneous wall reaction), no bulk reaction, $Pe=100$, $L=20$ ($\zeta_\text{out}=0.2$), mesh graded towards the wall. Mixing-cup concentration and local Sherwood number against the series.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from reactor import *

w_, m_ = graetz(30, 800)
msh = tube_mesh(20, 200, 20, grade=1.0); u = velocity(msh, "poiseuille")
sp = Species(msh, u, Pe=100.0, Da=0.0, wall="reactive"); r = sp.solve()
zs = np.array([1.0, 2.0, 5.0, 10.0, 19.0]); cm = sp.mixing_cup_at(zs); sh = sp.sherwood(zs)
ser = np.array([np.sum(m_ * np.exp(-w_ * z / 100)) for z in zs]); shs = np.array([np.sum(m_ * w_ * np.exp(-w_ * z / 100)) / np.sum(m_ * np.exp(-w_ * z / 100)) for z in zs])
for k in range(len(zs)):
    say(f"z {zs[k]:5.1f} (zeta {zs[k] / 100:.3f}): c_mix FE {cm[k]:.6f}, series {ser[k]:.6f} | Sh FE {sh[k]:.3f}, series {shs[k]:.3f}")
say(f"Sh_inf = w_1 = {w_[0]:.4f}; max c in the field {comm.allreduce(sp.c.x.array.max(), op=MPI.MAX):.4f}")

> **✏️ Your turn (5 min).** (i) Where does the FE result differ most from the series, and is that a numerical error? (ii) $Sh$ approaches 3.657 — from which $\zeta$ on would you call the flow "mass-transfer developed"?

> *your answer*

## 7 · LLM exercise: the Graetz problem  *(lecture §D3)*
Ask an LLM for a DOLFINx 0.11 script for mass transfer to a reactive tube wall ($c=0$) in fully developed laminar flow, axisymmetric, $Pe=UR/D=100$, $L=20R$, that prints the outlet mixing-cup concentration and the
local Sherwood number at the outlet, **checked** against the Graetz solution. Paste it below, run it with `-np 2`, and judge it against §6.

In [ ]:
%%fenicsx -np 2
# Paste the LLM's script here (keep the %%fenicsx line).

> *your verdict on the LLM's code*